**Eye-Tracking & Neurocognitive Intelligence | Eye-Tracking Foundations & Signal Analysis | Presented by: Reza Saadatyar (2026)**

**02 | Gap-Aware Gaze Preprocessing and Velocity**

![Conceptual workflow for stage 02](../../images/02_gap_aware_gaze_preprocessing.svg)

**Objective:** Produce a transparent valid-sample representation without bridging gaps and calculate velocity in dva/s.

**Input evidence:** Notebook 01 selected 10 participants, 40 source records and 2,778,008 samples from GazeBase Round 1, Session 1. This notebook reads its current saved stage CSV, not the original GazeBase archive.

**Notebook scope:** Process one complete record at a time to bound memory use. Preserve raw x/y, source identity, invalid samples and parser labels; construct a smoothed gaze representation and adjacent-sample velocity without interpolation or blink reconstruction.

**Execution status:** Stage 02 has not been run on this 10-participant source in the committed evidence. Settings and the conceptual workflow are not measured results.

**Execution status:** Source code prepared. No original-data results are claimed until this notebook is executed with the specified release.


**00. Colab Repository Setup**

In [ ]:
# ======================== 00. Colab Repository Setup =========================

# -------------------------- 1. Repository Checkout ---------------------------
from pathlib import Path
import subprocess

repo_url = "https://github.com/RezaSaadatyar/Eye-Tracking-Neurocognitive-Intelligence.git"
repo_path = Path("/content") / repo_url.rsplit("/", 1)[-1].removesuffix(".git")
if not repo_path.exists():
    subprocess.run(["git", "clone", "-q", repo_url, str(repo_path)], check=True)

# ---------------------------- 2. Open Repository -----------------------------
%cd {repo_path}


**01. Imports and Research Environment**

In [ ]:
# =================== 01. Imports and Research Environment ====================

# ------------------------ 1. Study Environment Setup -------------------------
from python_modules.core.colab_setup import setup_colab
root = setup_colab("eye_tracking_foundations")

# ---------------------------- 2. Standard Library ----------------------------
from pathlib import Path

# ----------------- 3. Scientific Computing and Data Analysis -----------------
import pandas as pd

# ------------------- 4. Visualisation and Notebook Display -------------------
import matplotlib.pyplot as plt
from IPython.display import display
from core.plot_style import plot_style, save_figure
from core.output_download import download_outputs_from_colab

# ------------------------ 5. Study Research Utilities ------------------------
from core.stage_outputs import ensure_stage_inputs
from gaze_streaming import preprocess_stage_file

# --------------------- 6. Output and Presentation Setup ----------------------
output_dir = root / "eye_tracking_foundations" / "output"
output_dir.mkdir(parents=True, exist_ok=True)
plot_style(
    font_family="DejaVu Sans", title_size=12.5,
    title_weight="normal", label_size=11.5,
    label_weight="normal", tick_size=11,
    legend_size=10, figure_dpi=100, figure_size=(10, 4),
)
print("Current outputs:", output_dir)


**02. Recover Notebook 01 Current Inputs**

Read the complete `01_gaze_samples.csv` produced by Notebook 01. The committed
Notebook 01 run recorded **10 participants, 40 recordings and 2,778,008 samples**
from Round 1, Session 1. The source manifest records the selected participant,
task and record identifiers; it is a compact control for checking coverage.

In the same Colab runtime, use the existing files in `output/`. For a new
runtime, upload `eye_tracking_foundations_Notebook_01_Outputs.zip` to
`/content/` before running this section. The stage helper restores named files;
there is **no raw GazeBase reread, new download or Notebook 01 recomputation**.
The large sample CSV is read in bounded chunks during the next stage.


In [ ]:
# ================== 02. Recover Notebook 01 Current Inputs ===================

# ----------------------- 1. Verify Upstream Outputs --------------------------
ensure_stage_inputs(
    output_dir, 1, ("01_gaze_samples.csv", "01_source_manifest.csv")
)
source_csv = output_dir / "01_gaze_samples.csv"
manifest = pd.read_csv(
    output_dir / "01_source_manifest.csv",
    dtype={"record_id": str, "participant_id": str}, low_memory=False,
)
if manifest["record_id"].duplicated().any():
    raise ValueError("Duplicate record identities in Notebook 01 manifest.")
expected_records = set(manifest["record_id"].astype(str))
print("Upstream participants:", manifest["participant_id"].nunique())
print("Upstream task recordings:", len(manifest))
print("Source CSV on disk (MiB):", round(source_csv.stat().st_size / 1024**2, 1))
display(manifest[["record_id", "participant_id", "task"]].head(8))


**03. Apply Gap-Respecting Smoothing**

The preprocessing method is applied **separately to each complete source
record**, reading the 2.78-million-row upstream CSV in chunks rather than holding
all rows and their transformed copies in memory. Record boundaries are checked
against the saved manifest; the scientific transformation is unchanged.

A centred **five-sample rolling median** is computed only within contiguous valid
gaze runs. A valid run requires source `val = 0`, finite `x/y`, increasing
timestamps and a gap of at most **5 ms** between adjacent samples. These are
provisional method settings, not parameters selected by reference labels.

Original coordinates, timestamps, sample/record identifiers, optional pupil
values and source parser labels remain in the saved output. No gap interpolation,
blink reconstruction or mixing of records occurs. A temporary file is replaced
atomically after all records have been processed successfully.


In [ ]:
# ==================== 03. Apply Gap-Respecting Smoothing =====================

# -------------------- 1. Declare Scientific Settings -------------------------
SMOOTHING_SAMPLES = 5
MAX_GAP_MS = 5.0
READ_CHUNK_ROWS = 80_000

# ------------------ 2. Stream Full Records and Save Output -------------------
segments, example_trace = preprocess_stage_file(
    source_csv,
    output_dir / "02_preprocessed_samples.csv",
    smoothing_samples=SMOOTHING_SAMPLES,
    max_gap_ms=MAX_GAP_MS,
    chunksize=READ_CHUNK_ROWS,
    expected_records=expected_records,
)
if len(segments) != len(manifest):
    raise ValueError("Preprocessing record count differs from the source manifest.")
if int(segments["n_samples"].sum()) <= 0:
    raise ValueError("No source samples were processed.")
print("Prepared records:", len(segments))
print("Prepared samples:", int(segments["n_samples"].sum()))


**04. Inspect Segments and Velocity**

Velocity is the Euclidean distance between consecutive **smoothed** valid gaze
coordinates divided by their actual timestamp difference, expressed in
degrees of visual angle per second (dva/s). The centred smoothing window uses
neighbouring samples within each run, so this is an **offline**, not a
real-time causal, representation.

Velocity is undefined at each valid run's first sample; invalid samples and
time gaps are not treated as zero motion. The summary reports valid samples,
segmentation and finite velocities for every recording. A single record is
processed in memory at a time.


In [ ]:
# ===================== 04. Inspect Segments and Velocity =====================

# ----------------------- 1. Persist Record Summary ---------------------------
segments.to_csv(output_dir / "02_preprocessing_summary.csv", index=False)
display(segments)

# -------------------- 2. Check Observable Coverage ---------------------------
print("Participants represented:", segments["participant_id"].nunique())
print("Source records represented:", segments["record_id"].nunique())
print("Prepared output (MiB):", round(
    (output_dir / "02_preprocessed_samples.csv").stat().st_size / 1024**2, 1
))


**05. Compare Raw and Processed Gaze**

Visualise up to the first 3,000 samples of one selected recording, retained
during streaming. The upper panel compares raw horizontal gaze with its
within-run centred-median representation; the lower panel shows the derived
angular speed, with undefined values preserved. The displayed plot is
**empirical only after this notebook runs** and should not be interpreted as
proof that smoothing improves fixation or saccade boundaries.


In [ ]:
# ==================== 05. Compare Raw and Processed Gaze =====================

record_id = str(example_trace.record_id.iloc[0])
part = example_trace
fig, axes = plt.subplots(2,1, figsize=(11,5), sharex=True)
axes[0].plot(part.time_s, part.x, lw=0.6, label="raw x")
axes[0].plot(part.time_s, part.x_smooth, lw=0.8, label="median-smoothed x")
axes[0].set_ylabel("Horizontal gaze (dva)")
axes[0].legend()
axes[1].plot(part.time_s, part.velocity_dva_s, lw=0.7)
axes[1].set_ylabel("Velocity (dva/s)")
axes[1].set_xlabel("Time from record start (s)")
fig.suptitle(f"Gap-aware motion representation: {record_id}")
fig.tight_layout()
save_figure(fig, output_dir / "02_velocity_profile.png")
plt.show()


**06. Export Reusable Numerical Inputs**

The current, stage-owned `02_preprocessed_samples.csv` contains the full
40-record sample-level representation and is the direct input to Notebook 03.
`02_preprocessing_summary.csv` contains record-level quality and velocity
statistics; `02_velocity_profile.png` records the inspected example.

The unchanged Human Movement download helper packages these named outputs for
manual saving. As the complete sample table may be large, keep the ZIP privately
and upload it only when continuing in a new Colab runtime. Do **not** commit
the generated CSV or ZIP to the public GitHub repository. Do not recompute
Notebook 01 or Notebook 02 when a compatible current output is available.


In [ ]:
# =================== 06. Export Reusable Numerical Inputs ====================

print("Missing velocities at segment starts remain NaN.")
print("Current motion representation:", output_dir / "02_preprocessed_samples.csv")
download_outputs_from_colab("eye_tracking_foundations/output", "02")
